# 00 — Construcción del dataset base

Construye `data/interim/habitia_madrid_2018.csv`, el dataset del que parten el EDA (01) y el
modelo predictivo de precios, a partir de las fuentes de `data/raw/`.

**Fuentes**
- `idealista18` — anuncios de venta 2018 ([paezha/idealista18](https://github.com/paezha/idealista18)).
  Rey-Blanco et al. (2024), *EPB: Urban Analytics and City Science*.
  (`data/raw/idealista18/`)
- Seccionado censal 2018 — ([INE](https://www.ine.es/dyngs/DAB/index.htm?cid=1389)) (`data/raw/ine_seccionado_2018/`).
- Mapeo sección censal → barrio — Ayuntamiento de Madrid (`data/raw/madrid_seccionado/`) ([2026 (JSON)](https://datos.madrid.es/dataset/300724-0-seccionado-censal-mapas/downloads)) e ([histórico (filtrado)](https://www.madrid.es/portales/munimadrid/es/Inicio/El-Ayuntamiento/Estadistica/Areas-de-informacion-estadistica/Territorio-y-medio-ambiente/Territorio/Seccionado-censal/?vgnextfmt=default&vgnextoid=c13456bc06f59210VgnVCM2000000c205a0aRCRD&vgnextchannel=e59b40ebd232a210VgnVCM1000000b205a0aRCRD)).
- Alquiler por sección censal — ([Ayuntamiento de Madrid](https://www.madrid.es/portales/munimadrid/es/Inicio/El-Ayuntamiento/Estadistica/Areas-de-informacion-estadistica/Edificacion-y-vivienda/Mercado-de-la-vivienda/Sistema-estatal-de-indices-de-referencia-del-precio-del-alquiler-de-viviendas/?vgnextfmt=default&vgnextoid=a386d9a1fe930910VgnVCM2000001f4a900aRCRD&vgnextchannel=22613c7ea422a210VgnVCM1000000b205a0aRCRD)) (`data/raw/madrid_alquiler/`).
- Equipamientos: centros educativos, parques, espacios deportivos y atención médica —
  [datos.madrid.es](https://datos.madrid.es/) (`data/raw/madrid_equipamientos/`).
- [Incidencias de Policía Municipal 2026](https://datos.madrid.es/dataset/837676-0-incidencias-recibidas-en-la-emisora-central-de-policia-municipal) y [población por barrio](https://datos.madrid.es/dataset/300557-0-poblacion-distrito-barrio) — Ayuntamiento de Madrid
  (`data/raw/madrid_incidencias_2026/`, `data/raw/madrid_poblacion/`).
- [Índice de vulnerabilidad por barrio 2017](https://datos.madrid.es/dataset/300301-0-ranking-vulnerabilidad) — Ayuntamiento de Madrid (`data/raw/madrid_vulnerabilidad/`).

**Salida:** un registro por anuncio, con `barrio_code` como unidad de análisis
geográfico y `codigo_censal` conservado como campo de procedencia para
enriquecer más adelante con fuentes que publican a nivel de sección censal
(INE Atlas de Renta, Censo).

---

## Correcciones metodológicas

Cuatro correcciones metodológicas, documentadas en su sección correspondiente:

| Problema | Sección |
|---|---|
| `LOCATIONCODE` de idealista **no es** una sección censal del INE | 4 |
| La sección censal (~2.440 unidades) es demasiado fina para 94.815 anuncios y para el ruido de anonimización del dataset — se pasa a `barrio` (131 unidades) como unidad de análisis | 4 |


## 1. Configuración

In [1]:
import os, sys, urllib.request, warnings
from pathlib import Path

RAIZ = Path.cwd()
while not (RAIZ / "src").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
os.chdir(RAIZ); sys.path.insert(0, str(RAIZ))

import geopandas as gpd
import numpy as np
import pandas as pd
import seaborn as sns
from shapely.geometry import shape
import json
from src import rutas
from src.aux_functions import mapa_censo_2026_a_2016

warnings.filterwarnings("ignore", category=UserWarning)

CRS_GEO = "EPSG:4326"      # WGS84 lon/lat — el CRS nativo de idealista18
CRS_UTM = 25830            # ETRS89 / UTM 30N — métrico, para distancias y áreas

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid")

## 2. Carga de `idealista18`

El paquete se distribuye solo para R. `rdata` lo lee, pero las columnas de
geometría (`sf`) llegan como listas anidadas y hay que reconstruirlas.

In [2]:
import rdata

ARCHIVOS = ["Madrid_Sale.rda", "Madrid_Polygons.rda", "Madrid_POIS.rda"]
URL_BASE = "https://raw.githubusercontent.com/paezha/idealista18/master/data"


def descargar():
    """Descarga los .rda de idealista18 que falten en data/raw/idealista18."""
    for nombre in ARCHIVOS:
        destino = rutas.DIR_IDEALISTA18 / nombre
        destino.parent.mkdir(parents=True, exist_ok=True)
        if not destino.exists():
            urllib.request.urlretrieve(f"{URL_BASE}/{nombre}", destino)
            print(f"descargado {nombre}")


def leer_rda(nombre, encoding=None):
    """Parsea un .rda y devuelve el primer objeto que contiene."""
    parsed = rdata.parser.parse_file(rutas.DIR_IDEALISTA18 / nombre)
    convertido = rdata.conversion.convert(parsed, default_encoding=encoding)
    return next(iter(convertido.values()))


descargar()
madrid_sale = leer_rda("Madrid_Sale.rda")
madrid_polygons = leer_rda("Madrid_Polygons.rda", encoding="utf8")

print(f"Anuncios:  {madrid_sale.shape}")
print(f"Polígonos: {madrid_polygons.shape}")

Anuncios:  (94815, 42)
Polígonos: (135, 4)


## Control de duplicados

In [3]:
print(f"Registros de inmuebles duplicados: {madrid_sale.duplicated(subset=['ASSETID']).sum()}")

Registros de inmuebles duplicados: 19011


### Nos quedamos únicamente con el registro del inmueble que tenga menor precio, el cual se entiende estará más cerca del precio de venta

**Limitación del modelo**: Los precios sobre los que se entrena el modelo pertenecen a este dataset de Idealista, donde aparecen precios anunciados. Cabe esperar que estos precios sean sustancialmente mayores a los precios a los que realmente se efectúe la compraventa. En consecuencia, los resultados del modelo, es decir, los precios predichos, se deberán entender como un sondeo de mercado, más que como una valoración de compraventa real.

In [4]:
madrid_sale = madrid_sale.loc[madrid_sale.groupby('ASSETID')['PRICE'].idxmin()]

print(f"Registros de inmuebles tras deduplicación: {len(madrid_sale)}")


Registros de inmuebles tras deduplicación: 75804


## 3. Reconstrucción de geometrías

Los puntos llegan como `[lon, lat]` y los polígonos como listas anidadas en
formato GeoJSON. Los puntos del fondo son `numpy.ndarray`, no listas — hay que
contemplarlo al detectar la profundidad de anidamiento.

In [5]:
def a_geometria(coords):
    """Lista anidada GeoJSON -> geometría shapely. Idempotente."""
    if hasattr(coords, "geom_type"):          # ya convertida
        return coords

    def profundidad(x):
        es_secuencia = isinstance(x, (list, tuple, np.ndarray))
        return 1 + profundidad(x[0]) if es_secuencia and len(x) else 0

    tipo = "MultiPolygon" if profundidad(coords) == 4 else "Polygon"
    return shape({"type": tipo, "coordinates": coords})


# --- Polígonos (zonas de idealista) ---
madrid_polygons["geometry"] = madrid_polygons["geometry"].apply(a_geometria)
zonas_gdf = gpd.GeoDataFrame(madrid_polygons, geometry="geometry", crs=CRS_GEO)

# --- Puntos (anuncios) ---
coords = np.stack(madrid_sale.geometry.to_numpy())
sale_gdf = gpd.GeoDataFrame(
    madrid_sale.drop(columns="geometry"),
    geometry=gpd.points_from_xy(coords[:, 0], coords[:, 1]),   # x=lon, y=lat
    crs=CRS_GEO,
)

# Validación: geometrías bien formadas y coordenadas en el rango de Madrid
#assert zonas_gdf.geometry.is_valid.all(), "hay polígonos inválidos"
#assert sale_gdf.geometry.x.between(-4.0, -3.4).all(), "longitudes fuera de rango"
#assert sale_gdf.geometry.y.between(40.2, 40.7).all(), "latitudes fuera de rango"

print(f"{len(sale_gdf):,} anuncios · {len(zonas_gdf)} zonas idealista")

75,804 anuncios · 135 zonas idealista


## 4. Backbone geográfico: asignación de sección censal

> ### Corrección 1 — `LOCATIONCODE` no es una sección censal
>
> La versión anterior derivaba un código a partir del `LOCATIONID` de idealista
> (`0-EU-ES-28-07-001-079-16-002` → `2807916002`) y lo unía contra
> `codigo_censal`. El código resultante **tiene el formato de un CUSEC pero no
> lo es**: los tres últimos dígitos de idealista identifican una *zona* dentro
> del distrito (van de 1 a 10), mientras que en el INE identifican una *sección
> censal* (van de 1 a 222).
>
> El resultado es un cruce arbitrario. Ejemplos verificados:
>
> | Zona idealista | Código | Ese código en el INE es |
> |---|---|---|
> | Conde Orgaz-Piovera | 2807916002 | Palomas |
> | Pinar del Rey | 2807916004 | Piovera |
> | Timón | 2807921004 | Alameda de Osuna |
> | Huertas-Cortes | 2807901003 | Palacio |
>
> Coinciden 119 de 135 códigos, así que el merge *parece* funcionar — pero une
> barrios con secciones censales que no les corresponden. Es un fallo silencioso:
> no lanza error y produce un dataset completo pero mal georreferenciado.
>
> **Solución:** asignar la sección censal por *join espacial* contra la
> cartografía del INE. Los anuncios tienen coordenadas; el INE publica los
> polígonos de sección. Ese es el cruce correcto.

In [6]:
from pathlib import Path


def localizar_shp(carpeta, patron="SECC_CE"):
    """
    Busca el .shp de secciones dentro de una carpeta ya descomprimida.
    Si hay varios .shp (líneas, puntos, distritos...), prioriza el que
    coincide con el patrón (p.ej. 'SECC_CE' o 'SECC_PA').
    """
    candidatos = sorted(Path(carpeta).rglob("*.shp"))
    if not candidatos:
        raise FileNotFoundError(f"No hay ningún .shp dentro de {carpeta}")

    preferidos = [c for c in candidatos if patron in c.stem.upper()]
    elegido = preferidos[0] if preferidos else candidatos[0]

    if len(candidatos) > 1:
        print(f"  {len(candidatos)} .shp encontrados en la carpeta, uso: {elegido.name}")
    return elegido


def cargar_secciones_ine(ano=2018, ruta_local=None, patron="SECC_CE"):
    """
    Seccionado censal del INE para Madrid capital, en CRS métrico.
    """
    ruta_local = Path(ruta_local or rutas.RAW / f"seccionado_{ano}")

    if not ruta_local.exists():
        raise FileNotFoundError(f"No encuentro {ruta_local}.")

    shp = ruta_local if ruta_local.suffix == ".shp" else localizar_shp(ruta_local, patron)

    gdf = gpd.read_file(shp)

    # OJO: no uppercase la columna de geometría, o geopandas pierde la
    # referencia a cuál es la geometría activa (KeyError: 'geometry' not in index)
    geom_col = gdf.geometry.name
    gdf.columns = [c.upper() if c != geom_col else c for c in gdf.columns]

    if "TIPO" in gdf.columns:                    # este fichero no lo trae; el filtro
        gdf = gdf[gdf.TIPO == "SECCION"]          # se queda por si otro año sí lo incluye

    gdf = gdf[gdf.CUSEC.str[:5] == "28079"]       # Madrid capital

    return (
        gdf[["CUSEC", "geometry"]]
        .rename(columns={"CUSEC": "codigo_censal"})
        .to_crs(CRS_UTM)
        .reset_index(drop=True)
    )


secciones = cargar_secciones_ine(2018, ruta_local=rutas.RUTA_SECCIONES_2018)
print(f"{len(secciones):,} secciones censales en Madrid capital")

2,443 secciones censales en Madrid capital


In [7]:
print(secciones.crs)
print(secciones.codigo_censal.str.len().unique())   # deben ser 10 dígitos
print(len(secciones))                                 # esperado: ~2.460 para Madrid capital

EPSG:25830
[10]
2443


#### La sección censal (~2.440 unidades) es demasiado fina para 94.815 anuncios + ruido de anonimización (91.5% de puntos a <77m de un borde de directamente contra barrio, no contra sección -> barrio).

In [8]:
# Corrección 4: 
with open(rutas.RUTA_MAPEO_SECCIONES, "r", encoding="utf8") as f:
    secciones_lookup_2026 = json.load(f)

secciones_lookup_2016 = mapa_censo_2026_a_2016(
    secciones_lookup_2026, rutas.RUTA_CAMBIOS_SECCIONES
)


def lookup_seccion(clave, campo):
    return secciones_lookup_2016.get(clave, {}).get(campo)


secciones["barrio_code"] = secciones["codigo_censal"].str[5:].map(lambda x: lookup_seccion(x, "COD_BAR"))
secciones["distrito_code"] = secciones["codigo_censal"].str[5:].map(lambda x: lookup_seccion(x, "COD_DIS"))

print(f"Secciones sin barrio mapeado: {secciones.barrio_code.isna().sum()} de {len(secciones)}")

barrios_gdf = (
    secciones.dropna(subset=["barrio_code"])
    .dissolve(by="barrio_code", as_index=False)[["barrio_code", "geometry"]]
)
print(f"{len(barrios_gdf)} barrios reconstruidos por disolución de secciones")

Secciones sin barrio mapeado: 0 de 2443
131 barrios reconstruidos por disolución de secciones


In [9]:
# Join espacial directo contra barrio (unidad de análisis), no contra sección
sale_utm = sale_gdf.to_crs(CRS_UTM)

datos = gpd.sjoin(sale_utm, barrios_gdf, how="left", predicate="within").drop(columns="index_right")
datos = datos[~datos.index.duplicated(keep="first")]

sin_barrio = datos.barrio_code.isna().sum()
print(f"Asignados: {datos.barrio_code.notna().sum():,} ({100*datos.barrio_code.notna().mean():.2f}%)")
print(f"Sin barrio: {sin_barrio:,}")

Asignados: 75,798 (99.99%)
Sin barrio: 6


### Rescate por proximidad para puntos fuera de todo barrio, entendidendo que existe el ruido de anonimización

In [10]:

if sin_barrio:
    faltan = datos.barrio_code.isna()
    rescate = gpd.sjoin_nearest(
        sale_utm.loc[faltan, ["geometry"]], barrios_gdf, how="left", max_distance=500,
    )
    rescate = rescate[~rescate.index.duplicated(keep="first")]
    datos.loc[faltan, "barrio_code"] = rescate.barrio_code


sin_asignar = datos.barrio_code.isna().sum()
if sin_asignar:
    print(f"Eliminando {sin_asignar} anuncio(s) sin barrio asignable ni por rescate.")
    datos = datos[datos.barrio_code.notna()]

print(f"Cobertura final: {100 * datos.barrio_code.notna().mean():.2f}%")

# distrito_code viaja con el barrio (relación N:1, ver dissolve)
distrito_por_barrio = (
    secciones.dropna(subset=["barrio_code"])
    .drop_duplicates("barrio_code")
    .set_index("barrio_code")["distrito_code"]
)
datos["distrito_code"] = datos.barrio_code.map(distrito_por_barrio)

NOMBRES_DISTRITO = {
    "01": "Centro", "02": "Arganzuela", "03": "Retiro", "04": "Salamanca",
    "05": "Chamartín", "06": "Tetuán", "07": "Chamberí",
    "08": "Fuencarral-El Pardo", "09": "Moncloa-Aravaca", "10": "Latina",
    "11": "Carabanchel", "12": "Usera", "13": "Puente de Vallecas",
    "14": "Moratalaz", "15": "Ciudad Lineal", "16": "Hortaleza",
    "17": "Villaverde", "18": "Villa de Vallecas", "19": "Vicálvaro",
    "20": "San Blas-Canillejas", "21": "Barajas",
}
datos["distrito"] = datos.distrito_code.map(NOMBRES_DISTRITO)

Eliminando 1 anuncio(s) sin barrio asignable ni por rescate.
Cobertura final: 100.00%


In [11]:
bordes_barrio = barrios_gdf.copy()
bordes_barrio["geometry"] = bordes_barrio.geometry.boundary

In [12]:
# codigo_censal: se conserva solo como campo de procedencia para cruces
# futuros a nivel de sección (INE Atlas de Renta, Censo). Join propio, más
# ligero, sin métrica de calidad -- ya no es la unidad de análisis.
cod_censal = gpd.sjoin(
    sale_utm[["geometry"]], secciones[["codigo_censal", "geometry"]],
    how="left", predicate="within",
).drop(columns="index_right")
cod_censal = cod_censal[~cod_censal.index.duplicated(keep="first")]

sin_seccion = cod_censal.codigo_censal.isna().sum()
print(f"Con codigo_censal: {cod_censal.codigo_censal.notna().sum():,} "
      f"({100*cod_censal.codigo_censal.notna().mean():.2f}%)")

Con codigo_censal: 75,798 (99.99%)


In [13]:
## Imputación de secciones faltantes por proximidad (500 m) a la sección más cercana

if sin_seccion:
    faltan_seccion = cod_censal.codigo_censal.isna()
    rescate_seccion = gpd.sjoin_nearest(
        sale_utm.loc[faltan_seccion, ["geometry"]], secciones[["codigo_censal", "geometry"]],
        how="left", max_distance=500,
    )
    rescate_seccion = rescate_seccion[~rescate_seccion.index.duplicated(keep="first")]
    cod_censal.loc[faltan_seccion, "codigo_censal"] = rescate_seccion.codigo_censal

datos["codigo_censal"] = cod_censal.reindex(datos.index).codigo_censal
print(f"Cobertura final de codigo_censal: {100*datos.codigo_censal.notna().mean():.2f}%")

Cobertura final de codigo_censal: 100.00%


## 5. Datos de alquiler por sección censal

Fuente: Ayuntamiento de Madrid, precios de alquiler por sección censal (2016). Fuente adjunta en la cabecera del notebook.
Dos años de *lag* dado que es el mismo que tendrán los datos en producción. Serán reagrupados por barrio, para evitar muestras pequeñas que causen volatilidad.

In [14]:
RUTA_ALQUILER = rutas.RUTA_ALQUILER

COLUMNAS_ALQUILER = {
    "Unnamed: 0": "codigo_censal",
    "Nº viviendas": "n_viviendas_alquiler",
    "Mediana.1": "alq_mediana_eur_m2_barrio",
}

COLS_PRECIO = [
    "alq_mediana_eur_m2_barrio"
]


def cargar_alquiler(hoja, ruta=RUTA_ALQUILER):
    """Carga y normaliza la tabla de alquiler por sección censal."""
    df = pd.read_excel(ruta, sheet_name=hoja, header=5)
    df = df[list(COLUMNAS_ALQUILER)].rename(columns=COLUMNAS_ALQUILER)

    # Filas de cabecera/pie: se identifican por no tener código numérico,
    # en lugar de por posición fija (más robusto ante cambios de formato)
    df = df[df.codigo_censal.astype(str).str.strip().str.match(r"^\d+$", na=False)]

    df["codigo_censal"] = "28079" + df.codigo_censal.astype(str).str.zfill(5)

    # El original usa coma decimal
    for col in COLS_PRECIO:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(",", "."), errors="coerce"
        )

    # Fila fantasma en el fichero fuente: un "2016" suelto en la columna de
    # código (resto del propio encabezado de hoja) que el regex admite como
    # código válido y que, tras el zfill, coincide por azar con una sección
    # real. No aporta dato -- todas las columnas de precio vienen NaN -- se
    # descarta explícitamente en vez de dejar que duplique una sección real.
    antes = len(df)
    df = df.dropna(subset=COLS_PRECIO, how="all")
    if len(df) < antes:
        print(f"  Descartada(s) {antes - len(df)} fila(s) sin ningún dato de precio")

    return df.reset_index(drop=True)


alquiler = cargar_alquiler(ruta=RUTA_ALQUILER, hoja="2016")
print(f"{len(alquiler):,} secciones con datos de alquiler")
print(f"Códigos únicos: {alquiler.codigo_censal.nunique():,}")

  Descartada(s) 66 fila(s) sin ningún dato de precio
2,377 secciones con datos de alquiler
Códigos únicos: 2,377


In [15]:
# La fuente publica a nivel de sección; se agrega a barrio (unidad de
# análisis) ponderando por n_viviendas_alquiler para no pesar igual una
# sección de 20 viviendas que una de 2.000.
alquiler["barrio_code"] = alquiler["codigo_censal"].str[5:].map(lambda x: lookup_seccion(x, "COD_BAR"))


def _agregado_ponderado(grupo, columnas, peso="n_viviendas_alquiler"):
    pesos = grupo[peso].fillna(0)
    valores = {}
    for col in columnas:
        valido = grupo[col].notna() & (pesos > 0)
        valores[col] = (
            np.average(grupo.loc[valido, col], weights=pesos.loc[valido])
            if valido.any() else np.nan
        )
    valores[peso] = pesos.sum()
    return pd.Series(valores)


alquiler_barrio = (
    alquiler.dropna(subset=["barrio_code"])
    .groupby("barrio_code", observed=True)
    .apply(lambda g: _agregado_ponderado(g, COLS_PRECIO), include_groups=False)
    .reset_index()
)

print(f"{len(alquiler_barrio)} barrios con datos de alquiler agregados")
print(f"Barrios sin ningún dato tras agregar: {alquiler_barrio.alq_mediana_eur_m2_barrio.isna().sum()}")

130 barrios con datos de alquiler agregados
Barrios sin ningún dato tras agregar: 0


### Verificación de llaves antes del merge

In [16]:
barrios_anuncios = set(datos.barrio_code.dropna())
barrios_alquiler = set(alquiler_barrio.barrio_code)

print(f"Barrios con anuncios:           {len(barrios_anuncios):,}")
print(f"Barrios con alquiler:           {len(barrios_alquiler):,}")
print(f"Intersección:                   {len(barrios_anuncios & barrios_alquiler):,}")
print(f"Con anuncios pero sin alquiler: {len(barrios_anuncios - barrios_alquiler):,}")

cobertura = len(barrios_anuncios & barrios_alquiler) / len(barrios_anuncios)
print(f"\nCobertura: {cobertura*100:.1f}% de los barrios con anuncios")

Barrios con anuncios:           131
Barrios con alquiler:           130
Intersección:                   130
Con anuncios pero sin alquiler: 1

Cobertura: 99.2% de los barrios con anuncios


In [17]:
datos = datos.merge(alquiler_barrio, on="barrio_code", how="left")

print(f"Filas: {len(datos):,}")
print(f"Con dato de alquiler: {datos.alq_mediana_eur_m2_barrio.notna().mean()*100:.1f}%")

Filas: 75,803
Con dato de alquiler: 99.6%


In [18]:
# Descartamos anuncios sin dato de alquiler. Son 334 anuncios reales (viviendas
# a la venta legítimas, no un artefacto del cruce) del barrio 194 (El
# Cañaveral). El motivo por el que no tienen alquiler asociado es que ese
# barrio, tal y como existe hoy, no tenía datos de mercado de alquiler en
# 2016:
#   - No existía como entidad separada: se creó el 17/11/2017 a partir de
#     parte del barrio 191 (Vicálvaro).
#   - Su sección madre (19-001) sí existía en 2016, pero con solo 10 viviendas
#     en alquiler -- el Ayuntamiento censura la mediana por muestra
#     insuficiente (aparece como "..").
#   - Su sección propia (19-052) ni siquiera se había creado todavía.
# No es un NaN por fallo de cruce ni recuperable por imputación fiable: no hay
# dato real de alquiler en 2016 para esa zona porque, a efectos de mercado de
# alquiler, la zona todavía no existía. Se decide descartar (no imputar) para
# no introducir un valor inventado en una variable que se usa como feature.
antes = len(datos)
datos = datos.dropna(subset=["alq_mediana_eur_m2_barrio"])
descartados = antes - len(datos)
print(f"Descartados {descartados} anuncios sin dato de alquiler ({100*descartados/antes:.2f}%)")
print(f"Filas restantes: {len(datos):,}")

Descartados 334 anuncios sin dato de alquiler (0.44%)
Filas restantes: 75,469


El panel derecho es tan importante como el izquierdo: un distrito con
rentabilidad alta y mucha imputación no es un hallazgo, es una zona sobre la
que hay poca información real. Conviene leer siempre los dos juntos.

## 6. Distancias a Puntos de Interés (POI)

Carga de data de proximidad a servicios públicos: colegios, parques,
centros médicos y espacios deportivos desde el Ayuntamiento de Madrid.
Estas variables de proximidad son *features* de accesibilidad/servicios para cada inmueble.

In [19]:
from src.poi_distances import enriquecer_con_distancias_poi

print("Enriqueciendo con distancias a POI...")
datos = enriquecer_con_distancias_poi(
    datos,
    data_dir=rutas.DIR_EQUIPAMIENTOS,
    crs_utm=CRS_UTM,
    tipos_poi=("centro_educativo", "parque", "espacio_deporte", "centro_medico"),
)
print("✓ Enriquecimiento completado")

Enriqueciendo con distancias a POI...
  Cargando centro_educativo... ✓ (1610 puntos, media 164 m)
  Cargando parque... ✓ (203 puntos, media 527 m)
  Cargando espacio_deporte... ✓ (707 puntos, media 324 m)
  Cargando centro_medico... ✓ (277 puntos, media 379 m)
✓ Enriquecimiento completado


## 7. Índice de criminalidad por barrio

Carga de datos de incidencias recibidas por la Policía Municipal de Madrid
(2026). La agregación y el cruce se hacen por barrio, que es el nivel de granularidad elegido para variables socioeconómicas. Variable de contexto
urbano/seguridad que puede afectar la percepción de valor de los inmuebles.

El indicador es creado a través de las incidencias reportadas en 2026 por la Policía Municipal, filtradas por delitos de cierta peligrosidad (ver función *enriquecer_con_incidencias*)

**Limitación del modelo**: Los datos de criminalidad están a niveles de 2026 porque la estadística no está disponible anteriormente. Por tanto, el modelo cuenta con la presuposición de que la distribución del crimen entre distritos en Madrid se ha mantenido estable entre 2018 y 2026. En efecto, esta premisa puede ser falsa y puede haber sesgado el modelo.

In [20]:
from src.crime_indices import enriquecer_con_incidencias

print("Enriqueciendo con incidencias ponderadas por poblacion...")
datos = enriquecer_con_incidencias(
    datos,
    data_dir=str(rutas.DIR_INCIDENCIAS),
    ruta_poblacion=str(rutas.RUTA_POBLACION),
    barrio_code_col="barrio_code",
)
print("✓ Enriquecimiento completado")

Enriqueciendo con incidencias ponderadas por poblacion...

Enriqueciendo con incidencias ponderadas por población...
  Cargando población por barrio...
  Cargando incidencias...
Se encontraron 6 archivos CSV

Cargando 837676-10-incidencias-recibidas-en-la-emisora-central-de-policia-municipal.csv...
Cargando 837676-12-incidencias-recibidas-en-la-emisora-central-de-policia-municipal.csv...
Cargando 837676-3-incidencias-recibidas-en-la-emisora-central-de-policia-municipal.csv...
Cargando 837676-4-incidencias-recibidas-en-la-emisora-central-de-policia-municipal.csv...
Cargando 837676-5-incidencias-recibidas-en-la-emisora-central-de-policia-municipal.csv...
Cargando 837676-8-incidencias-recibidas-en-la-emisora-central-de-policia-municipal.csv...
Total de registros cargados: 297567

Registros después del filtro por tipo: 32175
Tipos de incidencia únicos en datos filtrados: 9

INCIDENCIAS AGREGADAS POR BARRIO
Total de incidencias: 33244
Total de barrios: 131

  Barrios mapeados: 131/131
  Uni

## 8. Índice de vulnerabilidad por barrio

Carga de índice de vulnerabilidad socioeconómica de 2017 (Ayuntamiento de Madrid, 2018)
a nivel de barrio. Variable contextual que puede correlacionar con precio y
preferencias de demanda.

**Limitación del modelo**: El índice de vulnerabilidad que elabora ahora el Ayuntamiento tiene una magnitud distinta al elaborado en el año 2018. Por tanto, de manera similar al punto anterior, se simplificó la salida a producción del modelo con datos actuales utilizando este mismo índice, también entendiendo que la distribución relativa de problemática social por barrios no habrá cambiado significativamente. No obstante, esta hipótesis no es contrastada y, por tanto, esta decisión puede sesgar el modelo.

In [21]:
from src.vulnerability_indices import enriquecer_con_vulnerabilidad

print("Enriqueciendo con índice de vulnerabilidad...")
datos = enriquecer_con_vulnerabilidad(
    datos,
    ruta_vulnerabilidad=str(rutas.RUTA_VULNERABILIDAD),
    barrio_col="barrio_code",
)
print("✓ Enriquecimiento completado")

Enriqueciendo con índice de vulnerabilidad...
Cargando índice de vulnerabilidad...
  3 barrio(s) sin dato en 2018, cubiertos con 2019: 183, 193, 194
  Unida vulnerabilidad para 130 barrios
  Cobertura: 100.0%
  Media de índice de vulnerabilidad: 0.00792
✓ Enriquecimiento completado


**Apunte menor**: el ranking de 2018 no cubre `183` (Ensanche de Vallecas), `193`
(Valderrivas) ni `194` (El Cañaveral) — barrios demasiado recientes para esa
edición. `cargar_vulnerabilidad` cubre estos tres con su valor de 2019 (el año
más próximo en el que el Ayuntamiento ya los incluye) en vez de dejarlos en
NaN.

## 9. Validación del dataset final

In [22]:
def validar(df):
    """Comprobaciones de integridad antes de exportar. Si falla alguna, no se exporta."""
    lon = df.geometry.to_crs(CRS_GEO).x
    lat = df.geometry.to_crs(CRS_GEO).y
    checks = {
        "Sin filas duplicadas":
            not df.duplicated(subset=["ASSETID", "PERIOD"]).any(),
        "PRICE positivo":
            (df.PRICE > 0).all(),
        "Superficie positiva":
            (df.CONSTRUCTEDAREA > 0).all(),
        "codigo_censal de 10 dígitos":
            df.codigo_censal.dropna().str.len().eq(10).all(),
        "codigo_censal existe en el INE":
            df.codigo_censal.dropna().isin(secciones.codigo_censal).all(),
        "barrio_code y distrito_code asignados":
            df.barrio_code.notna().all() and df.distrito_code.notna().all(),
        "Coordenadas dentro de Madrid":
            lon.between(-4.0, -3.4).all() and lat.between(40.2, 40.7).all(),
    }

    for nombre, ok in checks.items():
        print(f"  {'✓' if ok else '✗'}  {nombre}")

    fallos = [nombre for nombre, ok in checks.items() if not ok]
    if fallos:
        raise AssertionError("Validación fallida, no se exporta: " + "; ".join(fallos))

    return True


print(f"Dataset final: {datos.shape[0]:,} filas × {datos.shape[1]} columnas\n")
validar(datos)

Dataset final: 75,469 filas × 56 columnas

  ✓  Sin filas duplicadas
  ✓  PRICE positivo
  ✓  Superficie positiva
  ✓  codigo_censal de 10 dígitos
  ✓  codigo_censal existe en el INE
  ✓  barrio_code y distrito_code asignados
  ✓  Coordenadas dentro de Madrid


True

In [23]:
# Resumen de cobertura por bloque de variables
bloques = {
    "Identificación": ["ASSETID", "PERIOD"],
    "Precio": ["PRICE", "UNITPRICE", "CONSTRUCTEDAREA"],
    "Geografía": ["barrio_code", "distrito_code", "distrito", "codigo_censal"],
    "Catastro": ["CADCONSTRUCTIONYEAR", "CADMAXBUILDINGFLOOR", "CADDWELLINGCOUNT"],
    "Alquiler": ["alq_mediana_eur_m2_barrio", "alq_estimado_eur"],
    "Proximidad a POI": ["dist_centro_educativo_m", "dist_parque_m", "dist_espacio_deporte_m", "dist_centro_medico_m"],
    "Seguridad": ["no_delitos_barrio", "poblacion_barrio", "delitos_per_10k_barrio"],
    "Vulnerabilidad": ["indice_vulnerabilidad"],
    "Calidad": ["bar_dudosa", "alq_imputado"],
}

resumen = pd.DataFrame([
    {
        "bloque": bloque,
        "variable": col,
        "cobertura_%": round(100 * datos[col].notna().mean(), 1),
    }
    for bloque, cols in bloques.items()
    for col in cols if col in datos.columns
])

resumen

,bloque,variable,cobertura_%
0,Identificación,ASSETID,100.0
1,Identificación,PERIOD,100.0
2,Precio,PRICE,100.0
3,Precio,UNITPRICE,100.0
4,Precio,CONSTRUCTEDAREA,100.0
5,Geografía,barrio_code,100.0
6,Geografía,distrito_code,100.0
7,Geografía,distrito,100.0
8,Geografía,codigo_censal,100.0
9,Catastro,CADCONSTRUCTIONYEAR,100.0


## 9.1 Claves de barrio y distrito

`barrio_code`/`distrito_code` no son *features* del modelo (ver `NON_EXP_COLS`
más abajo, sección de exportación), pero los notebooks posteriores los
necesitan para analizar el error por distrito (07) y para comprobar la asignación
de barrio en producción (08). Se guardan
aparte, indexados por `ASSETID`, antes de que la exportación los descarte.

In [24]:
SALIDA_GRUPOS = rutas.RUTA_ASSET_DISTRITO
SALIDA_GRUPOS.parent.mkdir(parents=True, exist_ok=True)

datos[["ASSETID", "barrio_code", "distrito_code"]].to_parquet(SALIDA_GRUPOS, index=False)
print(f"Guardado → {SALIDA_GRUPOS} "
      f"({datos['distrito_code'].nunique()} distritos, {datos['barrio_code'].nunique()} barrios)")

Guardado → C:\Users\tomas\Documents\Master\habitia-madrid\data\interim\asset_distrito.parquet (21 distritos, 130 barrios)


## 10. Exportación

Se guarda en CSV (separador `;`) en `data/interim/`, junto con un diccionario de datos
con el tipo y la cobertura de cada columna.

In [25]:
NON_EXP_COLS =   ["AMENITYID", 'no_delitos_barrio', 'poblacion_barrio',
                "LONGITUDE", "LATITUDE", "n_viviendas_alquiler", "barrio_code", "distrito_code", "distrito", "FLATLOCATIONID", "PERIOD", "geometry", "codigo_censal"]

datos.drop(columns=NON_EXP_COLS, inplace=True)

In [26]:
SALIDA = rutas.RUTA_DATASET_BASE
SALIDA.parent.mkdir(parents=True, exist_ok=True)

datos.to_csv(SALIDA, index=False, sep=";")
print(f"Guardado → {SALIDA}  ({SALIDA.stat().st_size / 1e6:.1f} MB)")

# Diccionario de datos para acompañar al dataset
diccionario = pd.DataFrame({
    "variable": datos.columns,
    "tipo": [str(t) for t in datos.dtypes],
    "n_nulos": datos.isna().sum().values,
    "cobertura_%": (100 * datos.notna().mean()).round(1).values,
})
diccionario.to_csv(rutas.RUTA_DICCIONARIO, index=False)
print(f"Diccionario → {rutas.RUTA_DICCIONARIO}")

Guardado → C:\Users\tomas\Documents\Master\habitia-madrid\data\interim\habitia_madrid_2018.csv  (16.9 MB)
Diccionario → C:\Users\tomas\Documents\Master\habitia-madrid\data\interim\diccionario_datos.csv
